# 🧠 03 — Model Training & Evaluation

## What are we doing here?

In the previous notebooks, we explored the data (EDA) and preprocessed it. Now comes the fun part — **training machine learning models** to detect network attacks!

### The Plan (in simple words):

1. **Load the data** — grab the NSL-KDD dataset we downloaded earlier
2. **Preprocess it** — scale numbers, encode categories (using our `preprocessing.py`)
3. **Train 6 different models** — each one "learns" to tell attacks from normal traffic
4. **Evaluate them** — see which model catches the most attacks without too many false alarms
5. **Save the best model** — so we can use it in our Streamlit dashboard later

### Two classification tasks:

| Task | Question it answers | Output |
|------|-------------------|--------|
| **Binary** | Is this traffic normal or an attack? | Normal (0) or Attack (1) |
| **Multi-class** | What *type* of attack is this? | Normal / DoS / Probe / R2L / U2R |

Let's get started! 🚀

---
## 🔧 Setup — Imports & Configuration

In [ ]:
# Standard libraries
import os
import sys
import time
import warnings
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Make plots look nice
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
warnings.filterwarnings('ignore')  # Keep the notebook clean

# Make sure we can import from src/
sys.path.insert(0, os.path.abspath('..'))

# Our custom modules
from src.data_loader import (
    load_train_data, load_test_data,
    add_attack_category, add_binary_label
)
from src.preprocessing import preprocess_data
from src.models import get_models, train_model, save_model
from src.evaluate import (
    evaluate_model, compare_models,
    print_classification_report
)

# sklearn metrics (we'll use these directly too)
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, f1_score, roc_auc_score
)

print("✅ All imports loaded successfully!")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

In [ ]:
# Create output directories
MODELS_DIR = os.path.join('..', 'models')
FIGURES_DIR = os.path.join('..', 'reports', 'figures')

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"📁 Models will be saved to: {os.path.abspath(MODELS_DIR)}")
print(f"📊 Figures will be saved to: {os.path.abspath(FIGURES_DIR)}")

---
## 📂 Step 1: Load the Data

We're using the **NSL-KDD** dataset — a standard benchmark for network intrusion detection.

**What's in it?**
- Each row = one network connection (e.g., someone connecting to a web server)
- 41 features describe that connection (duration, bytes sent, protocol, etc.)
- A label telling us if it was **normal** traffic or a specific **attack type**

The data is already split into train and test sets for us.

In [ ]:
# Load the raw datasets
train_df = load_train_data()
test_df = load_test_data()

# Add our label columns:
# - attack_category: Normal / DoS / Probe / R2L / U2R
# - binary_label: 0 (Normal) or 1 (Attack)
train_df = add_attack_category(train_df)
train_df = add_binary_label(train_df)
test_df = add_attack_category(test_df)
test_df = add_binary_label(test_df)

print(f"✅ Training data: {train_df.shape[0]:,} samples × {train_df.shape[1]} columns")
print(f"✅ Test data:     {test_df.shape[0]:,} samples × {test_df.shape[1]} columns")

In [ ]:
# Quick peek at the data
train_df.head()

In [ ]:
# How are the labels distributed?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Binary distribution
binary_counts = train_df['binary_label'].value_counts()
axes[0].bar(['Normal (0)', 'Attack (1)'], binary_counts.values,
            color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Binary Label Distribution (Train)')
axes[0].set_ylabel('Count')
for i, v in enumerate(binary_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Multi-class distribution
cat_counts = train_df['attack_category'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6']
axes[1].bar(cat_counts.index, cat_counts.values, color=colors)
axes[1].set_title('Attack Category Distribution (Train)')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(cat_counts.values):
    axes[1].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'label_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Notice: R2L and U2R attacks are rare — this is a class imbalance problem!")
print("   That's why we use 'class_weight=balanced' in some models.")

---
## 📦 Step 2: Preprocess the Data

Before we can train models, we need to **prepare** the data. Here's what happens:

| Step | What | Why |
|------|------|-----|
| **Scale numeric features** | Convert all numbers to a similar range (mean=0, std=1) | Some models (SVM, KNN, Logistic Regression) are sensitive to feature scales |
| **One-hot encode categories** | Turn `protocol_type=tcp` into `protocol_type_tcp=1, protocol_type_udp=0, ...` | ML models can't understand text — they need numbers |
| **Encode labels** | Turn "Normal"/"DoS"/etc. into 0/1/2/3/4 | Same reason — models need numbers for the target |

Our `preprocess_data()` function does all of this in one call! It fits the scaler on the **training data** and applies the same transformation to the **test data** (to avoid data leakage).

In [ ]:
# ========================================
# Preprocess for BINARY classification
# ========================================
# Target: binary_label (0 = Normal, 1 = Attack)

binary_data = preprocess_data(
    train_df, test_df,
    target_col='binary_label',
    label_type='binary'
)

X_train_bin = binary_data['X_train']
X_test_bin  = binary_data['X_test']
y_train_bin = binary_data['y_train']
y_test_bin  = binary_data['y_test']

print(f"✅ Binary preprocessing done!")
print(f"   X_train shape: {X_train_bin.shape}  (samples × features)")
print(f"   X_test shape:  {X_test_bin.shape}")
print(f"   Unique labels: {np.unique(y_train_bin)}  → [0=Normal, 1=Attack]")

In [ ]:
# ========================================
# Preprocess for MULTI-CLASS classification
# ========================================
# Target: attack_category (Normal / DoS / Probe / R2L / U2R)

multi_data = preprocess_data(
    train_df, test_df,
    target_col='attack_category',
    label_type='multiclass'
)

X_train_multi = multi_data['X_train']
X_test_multi  = multi_data['X_test']
y_train_multi = multi_data['y_train']
y_test_multi  = multi_data['y_test']
label_encoder = multi_data['label_encoder']

print(f"✅ Multi-class preprocessing done!")
print(f"   X_train shape: {X_train_multi.shape}")
print(f"   X_test shape:  {X_test_multi.shape}")
print(f"   Classes: {label_encoder.classes_}")
print(f"   Encoded as: {list(range(len(label_encoder.classes_)))}")

---
## 🏋️ Step 3: Train the Models

We'll train **6 different ML models**. Each has its own strengths:

| Model | How it works (simplified) | Strengths |
|-------|--------------------------|----------|
| **Logistic Regression** | Draws a line between classes using probabilities | Fast, interpretable, good baseline |
| **Decision Tree** | Asks yes/no questions about features (like a flowchart) | Easy to explain, fast |
| **Random Forest** | Combines 100 decision trees and takes a vote | Accurate, handles imbalanced data |
| **XGBoost** | Builds trees one at a time, each fixing the mistakes of the last | State-of-the-art for tabular data |
| **SVM** | Finds the widest "gap" between classes in high-dimensional space | Great for complex boundaries |
| **KNN** | Looks at the 5 nearest neighbors and copies their label | Simple, no training needed |

> 💡 **Why so many models?** Different algorithms have different biases. We try several and pick the one that works best on our data.

### 3A: Train Binary Models (Normal vs Attack)

In [ ]:
# Get our 6 models configured for binary classification
binary_models = get_models(task='binary')

print(f"🔧 Training {len(binary_models)} models for BINARY classification...")
print(f"   Task: Is this connection Normal or an Attack?\n")

# Train each model and measure how long it takes
binary_results = {}

for name, model in binary_models.items():
    print(f"   🔧 Training {name}...", end=" ", flush=True)
    result = train_model(model, X_train_bin, y_train_bin)
    print(f"✅ Done in {result['training_time']:.2f}s")
    binary_results[name] = result

print(f"\n🎉 All binary models trained!")

### 3B: Train Multi-class Models (5 attack categories)

In [ ]:
# Get our 6 models configured for multi-class classification
multi_models = get_models(task='multiclass')

print(f"🔧 Training {len(multi_models)} models for MULTI-CLASS classification...")
print(f"   Task: Which category is this? (Normal / DoS / Probe / R2L / U2R)\n")

# Train each model
multi_results = {}

for name, model in multi_models.items():
    print(f"   🔧 Training {name}...", end=" ", flush=True)
    result = train_model(model, X_train_multi, y_train_multi)
    print(f"✅ Done in {result['training_time']:.2f}s")
    multi_results[name] = result

print(f"\n🎉 All multi-class models trained!")

---
## 📊 Step 4: Evaluate & Compare Models

Now the important part — how well did each model perform? We'll check these metrics:

| Metric | What it means (simply) | Why it matters |
|--------|----------------------|---------------|
| **Accuracy** | % of correct predictions overall | Good overview, but misleading with imbalanced data |
| **Precision** | Of all connections flagged as attacks, how many really were? | High precision = fewer false alarms |
| **Recall** | Of all real attacks, how many did we catch? | High recall = fewer missed attacks |
| **F1 Score** | The balance between precision and recall | **Our main metric** — best of both worlds |
| **ROC-AUC** | How well the model separates classes at all thresholds | Higher = better overall discrimination |

> 💡 **F1 Score is our primary metric** because in security, we need both:
> - Not missing real attacks (recall)
> - Not flooding admins with false alarms (precision)

### 4A: Binary Model Comparison

In [ ]:
# Evaluate all binary models and create a comparison table
binary_comparison = compare_models(binary_results, X_test_bin, y_test_bin)

print("📊 BINARY MODEL COMPARISON (sorted by F1 Score)")
print("=" * 80)
binary_comparison.style.format({
    'accuracy': '{:.4f}',
    'precision': '{:.4f}',
    'recall': '{:.4f}',
    'f1_score': '{:.4f}',
    'roc_auc': '{:.4f}',
    'training_time': '{:.2f}s'
}).highlight_max(subset=['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc'], color='lightgreen')

In [ ]:
# Visual comparison — bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F1 Score comparison
colors = sns.color_palette('viridis', n_colors=len(binary_comparison))
bars = axes[0].barh(binary_comparison['model_name'], binary_comparison['f1_score'], color=colors)
for bar, val in zip(bars, binary_comparison['f1_score']):
    axes[0].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=10)
axes[0].set_xlabel('F1 Score')
axes[0].set_title('Binary Classification — F1 Score')
axes[0].set_xlim(0, 1.1)

# Training time comparison
bars2 = axes[1].barh(binary_comparison['model_name'], binary_comparison['training_time'], color=colors)
for bar, val in zip(bars2, binary_comparison['training_time']):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}s', va='center', fontsize=10)
axes[1].set_xlabel('Training Time (seconds)')
axes[1].set_title('Binary Classification — Training Time')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'binary_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

best_binary = binary_comparison.iloc[0]
print(f"\n🏆 Best binary model: {best_binary['model_name']} (F1 = {best_binary['f1_score']:.4f})")

### 4B: Multi-class Model Comparison

In [ ]:
# Evaluate all multi-class models
multi_comparison = compare_models(multi_results, X_test_multi, y_test_multi)

print("📊 MULTI-CLASS MODEL COMPARISON (sorted by F1 Score)")
print("=" * 80)
multi_comparison.style.format({
    'accuracy': '{:.4f}',
    'precision': '{:.4f}',
    'recall': '{:.4f}',
    'f1_score': '{:.4f}',
    'roc_auc': '{:.4f}',
    'training_time': '{:.2f}s'
}).highlight_max(subset=['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc'], color='lightgreen')

In [ ]:
# Visual comparison — bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F1 Score comparison
colors = sns.color_palette('viridis', n_colors=len(multi_comparison))
bars = axes[0].barh(multi_comparison['model_name'], multi_comparison['f1_score'], color=colors)
for bar, val in zip(bars, multi_comparison['f1_score']):
    axes[0].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=10)
axes[0].set_xlabel('F1 Score')
axes[0].set_title('Multi-class Classification — F1 Score')
axes[0].set_xlim(0, 1.1)

# Training time comparison
bars2 = axes[1].barh(multi_comparison['model_name'], multi_comparison['training_time'], color=colors)
for bar, val in zip(bars2, multi_comparison['training_time']):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}s', va='center', fontsize=10)
axes[1].set_xlabel('Training Time (seconds)')
axes[1].set_title('Multi-class Classification — Training Time')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'multiclass_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

best_multi = multi_comparison.iloc[0]
print(f"\n🏆 Best multi-class model: {best_multi['model_name']} (F1 = {best_multi['f1_score']:.4f})")

### 4C: Confusion Matrices

A **confusion matrix** shows exactly where the model gets confused:
- **Rows** = what the traffic actually was
- **Columns** = what the model predicted
- **Diagonal** = correct predictions ✅
- **Off-diagonal** = mistakes ❌

We want big numbers on the diagonal and small numbers everywhere else!

In [ ]:
# Confusion matrix for the BEST BINARY model
best_binary_name = binary_comparison.iloc[0]['model_name']
best_binary_model = binary_results[best_binary_name]['model']

y_pred_bin = best_binary_model.predict(X_test_bin)
cm_bin = confusion_matrix(y_test_bin, y_pred_bin)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {best_binary_name} (Binary)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f'binary_confusion_matrix_{best_binary_name.lower().replace(" ", "_")}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# Explain the numbers
tn, fp, fn, tp = cm_bin.ravel()
print(f"\n📖 Reading this matrix:")
print(f"   ✅ True Negatives (TN):  {tn:,} — Normal traffic correctly identified as normal")
print(f"   ❌ False Positives (FP): {fp:,} — Normal traffic wrongly flagged as attack (false alarm!)")
print(f"   ❌ False Negatives (FN): {fn:,} — Attacks that slipped through (missed attacks!)")
print(f"   ✅ True Positives (TP):  {tp:,} — Attacks correctly caught")

In [ ]:
# Confusion matrix for the BEST MULTI-CLASS model
best_multi_name = multi_comparison.iloc[0]['model_name']
best_multi_model = multi_results[best_multi_name]['model']

y_pred_multi = best_multi_model.predict(X_test_multi)
cm_multi = confusion_matrix(y_test_multi, y_pred_multi)

class_names = label_encoder.classes_.tolist()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {best_multi_name} (Multi-class)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f'multiclass_confusion_matrix_{best_multi_name.lower().replace(" ", "_")}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📖 Look at each row to see how well the model recognizes each attack type.")
print(f"   Big numbers on the diagonal = the model is good at that category.")
print(f"   R2L and U2R are usually hardest because they're so rare in the training data.")

### 4D: Detailed Classification Reports

This shows precision, recall, and F1 for **each class separately** — super useful for seeing which attack types the model struggles with.

In [ ]:
# Classification report for the best binary model
print(f"📋 Classification Report — {best_binary_name} (Binary)")
print("=" * 60)
print(classification_report(y_test_bin, y_pred_bin,
                           target_names=['Normal', 'Attack']))

In [ ]:
# Classification report for the best multi-class model
print(f"📋 Classification Report — {best_multi_name} (Multi-class)")
print("=" * 60)
print(classification_report(y_test_multi, y_pred_multi,
                           target_names=class_names))

### 4E: All Metrics at a Glance

Let's create a comprehensive heatmap showing all metrics for all models — makes it easy to compare everything at once.

In [ ]:
# Heatmap of all metrics for binary models
metrics_cols = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Binary heatmap
binary_heatmap_data = binary_comparison.set_index('model_name')[metrics_cols].astype(float)
sns.heatmap(binary_heatmap_data, annot=True, fmt='.4f', cmap='YlGn',
            linewidths=0.5, ax=axes[0], vmin=0.5, vmax=1.0)
axes[0].set_title('Binary Models — All Metrics')
axes[0].set_ylabel('')

# Multi-class heatmap
multi_heatmap_data = multi_comparison.set_index('model_name')[metrics_cols]
# Handle potential None values in roc_auc
multi_heatmap_data = multi_heatmap_data.apply(pd.to_numeric, errors='coerce')
sns.heatmap(multi_heatmap_data, annot=True, fmt='.4f', cmap='YlGn',
            linewidths=0.5, ax=axes[1], vmin=0.5, vmax=1.0)
axes[1].set_title('Multi-class Models — All Metrics')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'all_metrics_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 💾 Step 5: Save the Best Models

We save **three things** for each task:

1. **The model itself** — the trained ML algorithm
2. **The preprocessor** — so new data gets the same scaling/encoding
3. **The label encoder** (multi-class only) — to convert numbers back to category names

These saved files will be loaded by:
- `src/predict.py` — for making predictions in Python
- `app/app.py` — for the Streamlit web dashboard

In [ ]:
# ========================================
# Save the BEST BINARY model
# ========================================
best_binary_name = binary_comparison.iloc[0]['model_name']
best_binary_model = binary_results[best_binary_name]['model']

# Save model
binary_model_path = os.path.join(MODELS_DIR, 'binary_best_model.joblib')
save_model(best_binary_model, binary_model_path)

# Save preprocessor
binary_preprocessor_path = os.path.join(MODELS_DIR, 'binary_preprocessor.joblib')
joblib.dump(binary_data['preprocessor'], binary_preprocessor_path)

print(f"\n🏆 Best Binary Model: {best_binary_name}")
print(f"   💾 Model saved to: {binary_model_path}")
print(f"   💾 Preprocessor saved to: {binary_preprocessor_path}")

In [ ]:
# ========================================
# Save the BEST MULTI-CLASS model
# ========================================
best_multi_name = multi_comparison.iloc[0]['model_name']
best_multi_model = multi_results[best_multi_name]['model']

# Save model
multi_model_path = os.path.join(MODELS_DIR, 'multiclass_best_model.joblib')
save_model(best_multi_model, multi_model_path)

# Save preprocessor
multi_preprocessor_path = os.path.join(MODELS_DIR, 'multiclass_preprocessor.joblib')
joblib.dump(multi_data['preprocessor'], multi_preprocessor_path)

# Save label encoder (so we can convert numbers back to category names)
label_encoder_path = os.path.join(MODELS_DIR, 'multiclass_label_encoder.joblib')
joblib.dump(label_encoder, label_encoder_path)

print(f"\n🏆 Best Multi-class Model: {best_multi_name}")
print(f"   💾 Model saved to: {multi_model_path}")
print(f"   💾 Preprocessor saved to: {multi_preprocessor_path}")
print(f"   💾 Label encoder saved to: {label_encoder_path}")

In [ ]:
# ========================================
# Save all results as JSON (for reference)
# ========================================
all_results = {
    'binary': {
        'best_model': best_binary_name,
        'comparison': binary_comparison.to_dict(orient='records'),
    },
    'multiclass': {
        'best_model': best_multi_name,
        'comparison': multi_comparison.to_dict(orient='records'),
    }
}

results_path = os.path.join(MODELS_DIR, 'training_results.json')
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print(f"📄 Full results saved to: {results_path}")

In [ ]:
# ========================================
# Quick verification — load and test
# ========================================
print("🔍 Verifying saved models...\n")

# Load the binary model back
loaded_model = joblib.load(binary_model_path)
loaded_preprocessor = joblib.load(binary_preprocessor_path)

# Make a prediction on the first 5 test samples
sample_pred = loaded_model.predict(X_test_bin[:5])
print(f"✅ Binary model loaded and working!")
print(f"   Sample predictions: {sample_pred}")
print(f"   Actual labels:      {y_test_bin[:5]}")

# Load the multi-class model back
loaded_multi = joblib.load(multi_model_path)
loaded_le = joblib.load(label_encoder_path)

sample_pred_multi = loaded_multi.predict(X_test_multi[:5])
print(f"\n✅ Multi-class model loaded and working!")
print(f"   Sample predictions: {loaded_le.inverse_transform(sample_pred_multi)}")
print(f"   Actual labels:      {loaded_le.inverse_transform(y_test_multi[:5])}")

---
## 📝 Summary

### What we did in this notebook:

1. ✅ Loaded the NSL-KDD dataset (~126K train, ~22K test samples)
2. ✅ Preprocessed the data (scaled numeric features, one-hot encoded categories)
3. ✅ Trained 6 ML models for **binary** classification (Normal vs Attack)
4. ✅ Trained 6 ML models for **multi-class** classification (5 categories)
5. ✅ Evaluated all models with accuracy, precision, recall, F1, ROC-AUC
6. ✅ Generated confusion matrices and comparison charts
7. ✅ Saved the best models to `models/` directory

### Files created:

```
models/
├── binary_best_model.joblib
├── binary_preprocessor.joblib
├── multiclass_best_model.joblib
├── multiclass_preprocessor.joblib
├── multiclass_label_encoder.joblib
└── training_results.json

reports/figures/
├── label_distribution.png
├── binary_model_comparison.png
├── multiclass_model_comparison.png
├── binary_confusion_matrix_*.png
├── multiclass_confusion_matrix_*.png
└── all_metrics_heatmap.png
```

### Next steps:

- **Notebook 04**: Try deep learning (MLP neural network) with TensorFlow
- **Dashboard**: Build the Streamlit app using the saved models
- **Hyperparameter tuning**: Use GridSearch/RandomSearch to optimize the best model